In [2]:
from datasets import load_dataset
import pandas as pd
import json

print("Imports OK")

Imports OK


In [3]:
dataset = load_dataset("squad")
print(dataset)


README.md: 0.00B [00:00, ?B/s]

C:\Users\BrandlyFES\anaconda3\envs\DLxNLP\lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\BrandlyFES\.cache\huggingface\hub\datasets--squad. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/14.5M [00:00<?, ?B/s]

plain_text/validation-00000-of-00001.par(…):   0%|          | 0.00/1.82M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/87599 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/10570 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 87599
    })
    validation: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 10570
    })
})


In [4]:
exemple = dataset["train"][0]
print("Question :", exemple["question"])
print("Contexte :", exemple["context"][:300])
print("Réponse  :", exemple["answers"])

Question : To whom did the Virgin Mary allegedly appear in 1858 in Lourdes France?
Contexte : Architecturally, the school has a Catholic character. Atop the Main Building's gold dome is a golden statue of the Virgin Mary. Immediately in front of the Main Building and facing it, is a copper statue of Christ with arms upraised with the legend "Venite Ad Me Omnes". Next to the Main Building is 
Réponse  : {'text': ['Saint Bernadette Soubirous'], 'answer_start': [515]}


In [5]:
df_train = pd.DataFrame(dataset["train"])
print("Nombre d'exemples train :", len(df_train))
print("Colonnes :", df_train.columns.tolist())
print("\nAperçu :")
df_train[["question", "answers"]].head(5)

Nombre d'exemples train : 87599
Colonnes : ['id', 'title', 'context', 'question', 'answers']

Aperçu :


,question,answers
0,To whom did the Virgin Mary allegedly appear i...,"{'text': ['Saint Bernadette Soubirous'], 'answ..."
1,What is in front of the Notre Dame Main Building?,"{'text': ['a copper statue of Christ'], 'answe..."
2,The Basilica of the Sacred heart at Notre Dame...,"{'text': ['the Main Building'], 'answer_start'..."
3,What is the Grotto at Notre Dame?,{'text': ['a Marian place of prayer and reflec...
4,What sits on top of the Main Building at Notre...,{'text': ['a golden statue of the Virgin Mary'...


In [6]:
def extract_answer(answers):
    if len(answers["text"]) > 0:
        return answers["text"][0]
    return None

df_train["answer_text"] = df_train["answers"].apply(extract_answer)
df_train["answer_start"] = df_train["answers"].apply(lambda x: x["answer_start"][0] if x["answer_start"] else None)

# Garder seulement les colonnes utiles
df_clean = df_train[["id", "title", "context", "question", "answer_text", "answer_start"]].copy()

# Supprimer les lignes sans réponse
df_clean = df_clean.dropna(subset=["answer_text"])

print("Exemples après nettoyage :", len(df_clean))
print(df_clean.head(3))

Exemples après nettoyage : 87599
                         id                     title  \
0  5733be284776f41900661182  University_of_Notre_Dame   
1  5733be284776f4190066117f  University_of_Notre_Dame   
2  5733be284776f41900661180  University_of_Notre_Dame   

                                             context  \
0  Architecturally, the school has a Catholic cha...   
1  Architecturally, the school has a Catholic cha...   
2  Architecturally, the school has a Catholic cha...   

                                            question  \
0  To whom did the Virgin Mary allegedly appear i...   
1  What is in front of the Notre Dame Main Building?   
2  The Basilica of the Sacred heart at Notre Dame...   

                  answer_text  answer_start  
0  Saint Bernadette Soubirous           515  
1   a copper statue of Christ           188  
2           the Main Building           279  


In [7]:
df_val = pd.DataFrame(dataset["validation"])
df_val["answer_text"] = df_val["answers"].apply(extract_answer)
df_val["answer_start"] = df_val["answers"].apply(lambda x: x["answer_start"][0] if x["answer_start"] else None)
df_val_clean = df_val[["id", "title", "context", "question", "answer_text", "answer_start"]].dropna(subset=["answer_text"])

print("Validation exemples :", len(df_val_clean))

Validation exemples : 10570


In [8]:
df_clean.to_json("squad_train_clean.json", orient="records", force_ascii=False, indent=2)
df_val_clean.to_json("squad_val_clean.json", orient="records", force_ascii=False, indent=2)

print("Fichiers sauvegardés :")
print("  squad_train_clean.json —", len(df_clean), "exemples")
print("  squad_val_clean.json  —", len(df_val_clean), "exemples")

Fichiers sauvegardés :
  squad_train_clean.json — 87599 exemples
  squad_val_clean.json  — 10570 exemples


In [9]:
with open("squad_train_clean.json", "r", encoding="utf-8") as f:
    data = json.load(f)

print("Premier exemple propre :")
print("  Question :", data[0]["question"])
print("  Réponse  :", data[0]["answer_text"])
print("  Contexte :", data[0]["context"][:200])

Premier exemple propre :
  Question : To whom did the Virgin Mary allegedly appear in 1858 in Lourdes France?
  Réponse  : Saint Bernadette Soubirous
  Contexte : Architecturally, the school has a Catholic character. Atop the Main Building's gold dome is a golden statue of the Virgin Mary. Immediately in front of the Main Building and facing it, is a copper sta
